XGBOOST

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
import hashlib

In [ ]:
from google.colab import drive
# 1. Collega Drive
drive.mount('/content/drive')

# 2. Vai nella cartella dove hai i file e la cartella audio
%cd "/content/drive/MyDrive/Magistrale/Tesi/Fase2"

Mounted at /content/drive
/content/drive/MyDrive/Magistrale/Tesi/Fase2


In [ ]:
SEEDS = [42, 8, 1291, 64207, 305, 91876, 12, 456, 33920, 7]

In [ ]:
CSV_INPUT_PATH = f"audio_dataset.csv"
LABEL_COLUMN="portata"
LOW_HYPERPARAMETERS=False
NUMBER_OF_COMBINATIONS=50
MAIN_METRIC="MAE"
if MAIN_METRIC=="MSE":
  base_score_metric='neg_mean_squared_error'
else:
  base_score_metric='neg_mean_absolute_error'
START_FROM_SEED_INDEX=0 #0

In [ ]:
# 1. CARICAMENTO DATI
# Assumiamo che il file CSV abbia la colonna identificativa come prima colonna
df = pd.read_csv(CSV_INPUT_PATH, index_col=0)

In [ ]:
# 2. PREPARAZIONE X e y
target_column = 'portata'
X = df.drop(columns=[target_column])
y = df[target_column]

OTTIMIZZAZIONE IPERPARAMETRI

In [ ]:
param_distributions = {}

# 2. Definizione della griglia di iperparametri
if LOW_HYPERPARAMETERS:
  param_distributions = {
      'n_estimators': [5, 8, 10, 12, 15],
      'max_depth': [3, 4],             # Più realistici per XGBoost
      'learning_rate': [0.01, 0.05, 0.1, 0.2],       # Range classico (0.2 spesso è troppo aggressivo)
      'subsample': [0.8, 1.0],
      'colsample_bytree': [0.8, 1.0],
      'gamma': [0, 0.1, 0.5],                   # Aumentato il valore massimo per regolarizzare meglio
  }
else:
  param_distributions = {
      'n_estimators': [8, 12, 15, 20, 25],
      'max_depth': [4, 6, 9],                     # Più realistici per XGBoost
      'learning_rate': [0.01, 0.05, 0.1, 0.2],       # Range classico (0.2 spesso è troppo aggressivo)
      'subsample': [0.8, 1.0],
      'colsample_bytree': [0.8, 1.0],
      'gamma': [0, 0.1, 0.5],                   # Aumentato il valore massimo per regolarizzare meglio
  }

In [8]:
for i, seed in enumerate(SEEDS[START_FROM_SEED_INDEX:], start=START_FROM_SEED_INDEX):

  if START_FROM_SEED_INDEX != 0:
      print(f"Skipping the first {START_FROM_SEED_INDEX} seeds...")

  print(f"\n\n=== INIZIO RANDOMIZED SEARCH CON SEME {seed} ({i+1}/{len(SEEDS)}) ===\n")

  # 3. SPLIT TRAIN/TEST
  # Dividiamo i dati: 80% training, 20% test
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

  train_indices = df.index.get_indexer(X_train.index).tolist()
  test_indices = df.index.get_indexer(X_test.index).tolist()
  print("SHA256 Train set: "+hashlib.sha256(str(train_indices).encode('utf-8')).hexdigest())
  print("SHA256 Test set: "+hashlib.sha256(str(test_indices).encode('utf-8')).hexdigest())

  # 1. Definizione del modello base
  # Impostiamo verbosity=0 per evitare troppi log e random_state per la riproducibilità
  xgb_reg_to_optimize = xgb.XGBRegressor(base_score = 0, tree_method='hist', random_state=seed)

  # 3. Configurazione della Randomized Search
  randomized_search = RandomizedSearchCV(
      estimator=xgb_reg_to_optimize,
      param_distributions=param_distributions,
      n_iter=NUMBER_OF_COMBINATIONS,                    # Numero di combinazioni da testare
      cv=3,                         # Cross-validation a 3 fold
      scoring=base_score_metric,
      refit=False,
      random_state=42,
      verbose=2,
  )

  # 4. Esecuzione (Assumendo che X e y siano già pronti e scalati)
  randomized_search.fit(X_train, y_train)
  print(f"Randomized Search completato con seme {seed}")

  # Risultati
  print("Migliori parametri individuati:")
  print(randomized_search.best_params_)

  results = pd.DataFrame(randomized_search.cv_results_)
  results = results.sort_values(by="rank_test_score", ascending=True)
  RESULTS_CSV_PATH=f"Results_randomized_search_xgboost_{seed}.csv"
  results.to_csv(RESULTS_CSV_PATH, index=False)



=== INIZIO RANDOMIZED SEARCH CON SEME 42 (1/10) ===

SHA256 Train set: 72d2e222075a8f8f1af5150136c80e70a1597099b54f860bb98ff9aa9b9de413
SHA256 Test set: c942b7fdd659fe737f68c6b07db1ce15dee591ea6dc65f83c9d58ac3038fa68c
Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.2, max_depth=6, n_estimators=8, subsample=0.8; total time=   3.6s
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.2, max_depth=6, n_estimators=8, subsample=0.8; total time=   4.3s
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.2, max_depth=6, n_estimators=8, subsample=0.8; total time=   5.3s
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.05, max_depth=9, n_estimators=8, subsample=0.8; total time=   3.9s
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.05, max_depth=9, n_estimators=8, subsample=0.8; total time=   4.4s
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.05, max_depth=9, n_estimators=8, subsa

DEFINIZIONE DEL MODELLO FINALE

In [9]:
'''
#xgb_regressor = grid_search.best_estimator_
best_params = randomized_search.best_params_
xgb_regressor = xgb.XGBRegressor(**best_params, base_score = 0, random_state=SEED)

# 4. Addestramento finale sul set completo
xgb_regressor.fit(X_train, y_train)
print("Addestramento finale completato")
'''

'\n#xgb_regressor = grid_search.best_estimator_\nbest_params = randomized_search.best_params_\nxgb_regressor = xgb.XGBRegressor(**best_params, base_score = 0, random_state=SEED)\n\n# 4. Addestramento finale sul set completo\nxgb_regressor.fit(X_train, y_train)\nprint("Addestramento finale completato")\n'

VALUTAZIONE DEL MODELLO FINALE

In [10]:
'''
# 3. PREDIZIONE E VALUTAZIONE
y_pred = xgb_regressor.predict(X_test)
'''

'\n# 3. PREDIZIONE E VALUTAZIONE\ny_pred = xgb_regressor.predict(X_test)\n'

In [11]:
'''
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
r2 = r2_score(y_test, y_pred)
mse=mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"--- PERFORMANCE XGBOOST ---")
print(f"R^2 Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
'''

'\nmae = mean_absolute_error(y_test, y_pred)\nmape = mean_absolute_percentage_error(y_test, y_pred) * 100\nr2 = r2_score(y_test, y_pred)\nmse=mean_squared_error(y_test, y_pred)\nrmse = np.sqrt(mse)\n\nprint(f"--- PERFORMANCE XGBOOST ---")\nprint(f"R^2 Score: {r2:.4f}")\nprint(f"MAE: {mae:.4f}")\nprint(f"MAPE: {mape:.2f}%")\nprint(f"MSE: {mse:.4f}")\nprint(f"RMSE: {rmse:.4f}")\n'

In [12]:
'''
# VISUALIZZAZIONE: FEATURE IMPORTANCE
plt.figure(figsize=(10, 6))
importances = pd.Series(xgb_regressor.feature_importances_, index=X.columns)
importances.nlargest(10).sort_values(ascending=True).plot(kind='barh', color='orange')
plt.title("Top 10 Feature per la stima della Portata")
plt.xlabel("Importanza Relativa")
plt.tight_layout()
plt.show()
'''

'\n# VISUALIZZAZIONE: FEATURE IMPORTANCE\nplt.figure(figsize=(10, 6))\nimportances = pd.Series(xgb_regressor.feature_importances_, index=X.columns)\nimportances.nlargest(10).sort_values(ascending=True).plot(kind=\'barh\', color=\'orange\')\nplt.title("Top 10 Feature per la stima della Portata")\nplt.xlabel("Importanza Relativa")\nplt.tight_layout()\nplt.show()\n'